<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.6：生成器：类型
**上一步：[面向对象编程](3.5_object_oriented_programming.ipynb)**<br>
**下一步：[FIRRTL 简介](4.1_firrtl_ast.ipynb)**

## 动机
Scala 是一种强类型编程语言。
这是一把双刃剑；一方面，许多在 Python（一种动态类型语言）中可以编译和执行的程序在 Scala 中会在编译时失败。
另一方面，在 Scala 中编译的程序将比类似的 Python 程序包含更少的运行时错误。

在本节中，我们的目标是让您熟悉作为 Scala 中一等公民的类型。
虽然最初您可能会觉得工作效率有限，但您很快就会学会理解编译时错误消息，以及如何将类型系统牢记在心来构建程序，从而为您捕获更多错误。


## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 静态类型<a name="types-in-scala"></a>

## Scala 中的类型

Scala 中的所有对象都有一个类型，通常是该对象的类。
让我们看一些例子：

In [ ]:
println(10.getClass)
println(10.0.getClass)
println("ten".getClass)

当您声明自己的类时，它具有关联的类型。

In [ ]:
class MyClass {
    def myMethod = ???
}
println(new MyClass().getClass)

虽然不是必需的，但强烈建议您**为所有函数声明定义输入和输出类型**。
这将使 Scala 编译器能够捕获函数的错误使用。

In [ ]:
def double(s: String): String = s + s
// 取消注释下面的代码进行测试
// double("hi")      // 正确使用 double
// double(10)        // 错误的输入参数！
// double("hi") / 10 // double 输出的错误使用！

不返回任何内容的函数返回类型 `Unit`。

In [ ]:
var counter = 0
def increment(): Unit = {
    counter += 1
}
increment()

## Scala 类型与 Chisel 类型<a name="scala-vs-chisel-types"></a>

回顾：模块 2.2 讨论了 Chisel 类型和 Scala 类型之间的区别，例如以下事实：
```scala
val a = Wire(UInt(4.W))
a := 0.U
```
是合法的，因为 `0.U` 的类型是 `UInt`（一个 Chisel 类型），而
```scala
val a = Wire(UInt(4.W))
a := 0
```
是非法的，因为 0 的类型是 `Int`（一个 Scala 类型）。

对于 `Bool`（一个 Chisel 类型，与 `Boolean` 不同）也是如此。
```scala
val bool = Wire(Bool())
val boolean: Boolean = false
// 合法
when (bool) { ... }
if (boolean) { ... }
// 非法
if (bool) { ... }
when (boolean) { ... }
```

如果您犯了错误并混淆了 `UInt` 和 `Int` 或 `Bool` 和 `Boolean`，Scala 编译器通常会为您捕获它。
这是因为 Scala 的静态类型检查。
在编译时，编译器能够区分 Chisel 和 Scala 类型，并且还能够理解 `if ()` 期望一个 `Boolean` 而 `when ()` 期望一个 `Bool`。


## Scala 类型强制转换<a name="type-coercion"></a>

<!-- typeOf. Scala 有一个名为 `typeOf[T]` 的函数，它返回 `T` 的类型对象。 -->
<!-- 这对于 Chisel 用户来说似乎并不实用…… -->

### asInstanceOf

`x.asInstanceOf[T]` 将对象 `x` 强制转换为类型 `T`。如果给定对象无法强制转换为类型 `T`，则会引发异常。

In [ ]:
val x: UInt = 3.U
try {
  println(x.asInstanceOf[Int])
} catch {
  case e: java.lang.ClassCastException => println("正如预期的那样，我们无法将 UInt 转换为 Int")
}

// 但我们可以将 UInt 转换为 Data，因为 UInt 继承自 Data。
println(x.asInstanceOf[Data])


### Chisel 中的类型转换

如果您尝试在不删除注释的情况下运行以下代码，则会出错。
问题出在哪里？
它试图将 `UInt` 赋给 `SInt`，这是非法的。

Chisel 有一组类型转换函数。
最通用的是 `asTypeOf()`，如下所示。
一些 chisel 对象还定义了 `asUInt()` 和 `asSInt()` 以及其他一些函数。

如果您从下面的代码块中删除 `//`，则该示例应该可以正常工作。


In [ ]:
class TypeConvertDemo extends Module {
    val io = IO(new Bundle {
        val in  = Input(UInt(4.W))
        val out = Output(SInt(4.W))
    })
    io.out := io.in//.asTypeOf(io.out)
}

test(new TypeConvertDemo) { c =>
      c.io.in.poke(3.U)
      c.io.out.expect(3.S)
      c.io.in.poke(15.U)
      c.io.out.expect(-1.S)
}

---
# 类型匹配<a name="type-matching"></a>

## Match 运算符
回想一下，在 3.1 中介绍了 match 运算符。
在尝试编写类型通用生成器时，类型匹配尤其有用。
以下示例显示了一个可以添加两个类型为 `UInt` 或 `SInt` 的字面量的“生成器”示例。
后续章节将更多地讨论编写类型通用生成器。

**注意：在 Scala 中有更好、更安全的方法来编写类型通用生成器**。

In [ ]:
class ConstantSum(in1: Data, in2: Data) extends Module {
    val io = IO(new Bundle {
        val out = Output(chiselTypeOf(in1)) // 如果 in1 是字面量，则只获取其类型
    })
    (in1, in2) match {
        case (x: UInt, y: UInt) => io.out := x + y
        case (x: SInt, y: SInt) => io.out := x + y
        case _ => throw new Exception("我放弃了！")
    }
}
println(getVerilog(dut = new ConstantSum(3.U, 4.U)))
println(getVerilog(dut = new ConstantSum(-3.S, 4.S)))
println(getVerilog(dut = new ConstantSum(3.U, 4.S))) // 这会抛出异常


记住 Chisel 类型通常不应进行值匹配，这一点很重要。
Scala 的 match 在电路细化期间执行，但您可能想要的是细化后的比较。
以下代码会产生语法错误：

In [ ]:
class InputIsZero extends Module {
    val io = IO(new Bundle {
        val in  = Input(UInt(16.W))
        val out = Output(Bool())
    })
    io.out := (io.in match {
        // 注意 case 0.U 是一个错误
        case (0.U) => true.B
        case _   => false.B
    })
}
println(getVerilog(new InputIsZero))

## Unapply
当你进行匹配时，实际发生了什么？
Scala 如何让你像这样对 case class 进行花哨的值匹配：
```scala
case class Something(a: String, b: Int)
val a = Something("A", 3)
a match {
    case Something("A", value) => value
    case Something(str, 3)     => 0
}
```

事实证明，为每个 case class 创建的伴生对象还包含一个 **unapply** 方法，此外还有一个 **apply** 方法。
什么是 **unapply** 方法？

Scala unapply 方法是另一种语法糖形式，它赋予 match 语句在匹配期间既能匹配类型又能从这些类型中**提取值**的能力。

让我们看下面的例子。
出于某种原因，假设如果生成器正在流水线化，则延迟为 `3*totalWidth`，否则延迟为 `2*someOtherWidth`。
由于 case class 定义了 **unapply**，我们可以像这样匹配 case class 内部的值：

In [ ]:
case class SomeGeneratorParameters(
    someWidth: Int,
    someOtherWidth: Int = 10,
    pipelineMe: Boolean = false
) {
    require(someWidth >= 0)
    require(someOtherWidth >= 0)
    val totalWidth = someWidth + someOtherWidth
}

def delay(p: SomeGeneratorParameters): Int = p match {
    case SomeGeneratorParameters(_, sw, false) => sw * 2
    case sg @ SomeGeneratorParameters(_, _, true) => sg.totalWidth * 3
}

println(delay(SomeGeneratorParameters(10, 10)))
println(delay(SomeGeneratorParameters(10, 10, true)))

如果您查看 `delay` 函数，您应该注意到，除了匹配每个字符的类型之外，我们还：
- 直接引用参数的内部值
- 有时，直接匹配参数的内部值

由于编译器实现了 `unapply` 方法，这些都是可能的。请注意，unapply case 只是语法糖；例如，以下两个 case 示例是等效的：
```scala
case p: SomeGeneratorParameters => p.sw * 2
case SomeGeneratorParameters(_, sw, _) => sw * 2
```

此外，还有更多匹配的语法和样式。以下两种情况也是等效的，但第二种允许您在匹配内部值的同时仍然引用父值：
```scala
case SomeGeneratorParameters(_, sw, true) => sw
case sg @ SomeGeneratorParameters(_, sw, true) => sw
```

最后，您可以将条件检查直接嵌入到 match 语句中，如这三个等效示例中的第三个所示：
```scala
case SomeGeneratorParameters(_, sw, false) => sw * 2
case s @ SomeGeneratorParameters(_, sw, false) => s.sw * 2
case s: SomeGeneratorParameters if s.pipelineMe => s.sw * 2
```

所有这些语法都由类的伴生对象中包含的 Scala unapply 方法启用。如果您想 unapply 一个类但不想将其设为 case 类，则可以手动实现 unapply 方法。以下示例演示了如何手动实现类的 apply 和 unapply 方法：

In [ ]:
class Boat(val name: String, val length: Int)
object Boat {
    def unapply(b: Boat): Option[(String, Int)] = Some((b.name, b.length))
    def apply(name: String, length: Int): Boat = new Boat(name, length)
}

def getSmallBoats(seq: Seq[Boat]): Seq[Boat] = seq.filter { b =>
    b match {
        case Boat(_, length) if length < 60 => true
        case Boat(_, _) => false
    }
}

val boats = Seq(Boat("Santa Maria", 62), Boat("Pinta", 56), Boat("Nina", 50))
println(getSmallBoats(boats).map(_.name).mkString(" and ") + " are small boats!")

## 偏函数
这是一个简要概述；[本指南](https://twitter.github.io/scala_school/pattern-matching-and-functional-composition.html#PartialFunction) 提供了更详细的概述。

偏函数是仅在其输入的子集上定义的函数。
与选项类似，偏函数可能没有特定输入的值。
可以使用 `isDefinedAt(...)` 对此进行测试。

偏函数可以使用 `orElse` 链接在一起。

请注意，使用未定义的输入调用 `PartialFunction` 将导致运行时错误。例如，如果 `PartialFunction` 的输入是用户定义的，则可能会发生这种情况。为了更安全地进行类型检查，我们建议编写返回 `Option` 的函数。

In [ ]:
// 辅助函数，使此单元格不那么冗长。
def printAndAssert(cmd: String, result: Boolean, expected: Boolean): Unit = {
  println(s"$cmd = $result")
  assert(result == expected)
}

// 为 -1、2、5 等定义。
val partialFunc1: PartialFunction[Int, String] = {
  case i if (i + 1) % 3 == 0 => "Something"
}
printAndAssert("partialFunc1.isDefinedAt(2)", partialFunc1.isDefinedAt(2), true)
printAndAssert("partialFunc1.isDefinedAt(5)", partialFunc1.isDefinedAt(5), true)
printAndAssert("partialFunc1.isDefinedAt(1)", partialFunc1.isDefinedAt(1), false)
printAndAssert("partialFunc1.isDefinedAt(0)", partialFunc1.isDefinedAt(0), false)
println(s"partialFunc1(2) = ${partialFunc1(2)}")
try {
  println(partialFunc1(0))
} catch {
  case e: scala.MatchError => println("partialFunc1(0) = 无法在未定义它们的地方应用偏函数")
}

// 为 1、4、7 等定义。
val partialFunc2: PartialFunction[Int, String] = {
  case i if (i + 2) % 3 == 0 => "Something else"
}
printAndAssert("partialFunc2.isDefinedAt(1)", partialFunc2.isDefinedAt(1), true)
printAndAssert("partialFunc2.isDefinedAt(0)", partialFunc2.isDefinedAt(0), false)
println(s"partialFunc2(1) = ${partialFunc2(1)}")
try {
  println(partialFunc2(0))
} catch {
  case e: scala.MatchError => println("partialFunc2(0) = 无法在未定义它们的地方应用偏函数")
}

val partialFunc3 = partialFunc1 orElse partialFunc2
printAndAssert("partialFunc3.isDefinedAt(0)", partialFunc3.isDefinedAt(0), false)
printAndAssert("partialFunc3.isDefinedAt(1)", partialFunc3.isDefinedAt(1), true)
printAndAssert("partialFunc3.isDefinedAt(2)", partialFunc3.isDefinedAt(2), true)
printAndAssert("partialFunc3.isDefinedAt(3)", partialFunc3.isDefinedAt(3), false)
println(s"partialFunc3(1) = ${partialFunc3(1)}")
println(s"partialFunc3(2) = ${partialFunc3(2)}")

---
# 类型安全的连接<a name="type-safe-connections"></a>

Chisel 可以检查许多连接的类型，包括：
* Bool/UInt 到 Clock

对于其他类型，Chisel 会允许您连接它们，但可能会酌情截断/填充位。
* Bool/UInt 到 Bool/UInt
* Bundle 到 Bundle

In [ ]:
class Bundle1 extends Bundle {
  val a = UInt(8.W)
}

class Bundle2 extends Bundle1 {
  val b = UInt(16.W)
}

class BadTypeModule extends Module {
  val io = IO(new Bundle {
    val c  = Input(Clock())
    val in = Input(UInt(2.W))
    val out = Output(Bool())

    val bundleIn = Input(new Bundle2)
    val bundleOut = Output(new Bundle1)
  })
  
  //io.out := io.c // 由于类型不同而无法工作

  // 可以，但 Chisel 会将输入宽度截断为 1 以匹配输出。
//   io.out := io.in

//   // 编译通过；Chisel 将连接两个 Bundle 的公共子元素（在本例中为 'a'）。
//   io.bundleOut := io.bundleIn
}

println(getVerilog(new BadTypeModule))

---
# 类型泛型<a name="type-generics"></a>
Scala 的泛型类型（也称为多态性）非常复杂，尤其是在与继承结合使用时。

本节仅作初步介绍；要了解更多信息，请查看[本教程](https://twitter.github.io/scala_school/type-basics.html)。

类可以是类型多态的。序列就是一个很好的例子，它需要知道其包含的类型。

In [ ]:
val seq1 = Seq("1", "2", "3") // 类型为 Seq[String]
val seq2 = Seq(1, 2, 3)       // 类型为 Seq[Int]
val seq3 = Seq(1, "2", true)  // 类型为 Seq[Any]

有时，Scala 编译器需要帮助来确定多态类型，这需要用户显式地指定类型：

In [ ]:
//val default = Seq() // 错误！
val default = Seq[String]() // 用户必须告诉编译器 default 的类型是 Seq[String]
Seq(1, "2", true).foldLeft(default){ (strings, next) =>
    next match {
        case s: String => strings ++ Seq(s)
        case _ => strings
    }
}

函数也可以在其输入或输出类型上是多态的。以下示例定义了一个函数，该函数计算运行一段代码所需的时间。它根据代码块的返回类型进行参数化。*请注意，`=> T` 语法编码了一个没有参数列表的匿名函数，例如 `{ ... }` 与 `{ x => ... }`。*

In [ ]:
def time[T](block: => T): T = {
    val t0 = System.nanoTime()
    val result = block
    val t1 = System.nanoTime()
    val timeMillis = (t1 - t0) / 1000000.0
    println(s"代码块耗时 $timeMillis 毫秒！")
    result
}

// 将 1 加到一百万
val int = time { (1 to 1000000).reduce(_ + _) }
println(s"从 1 加到一百万的结果是 $int")

// 查找小于一百万且十六进制表示包含“beef”的最大数字
val string = time {
    (1 to 1000000).map(_.toHexString).filter(_.contains("beef")).last
}
println(s"小于一百万且包含 beef 的最大数字是：$string")

## Chisel 类型层次结构
要使用 Chisel 编写类型通用的代码，了解一些 Chisel 的类型层次结构会很有帮助。

`chisel3.Data` 是 Chisel 硬件类型的基类。
`UInt`、`SInt`、`Vec`、`Bundle` 等都是 `Data` 的实例。
`Data` 可以用于 IO，并支持 `:=`、连线、寄存器等。

寄存器是 Chisel 中多态代码的一个很好的例子。
请在此处查看 `RegEnable`（一个带有 `Bool` 使能信号的寄存器）的实现 [此处](https://github.com/freechipsproject/chisel3/blob/v3.0.0/src/main/scala/chisel3/util/Reg.scala#L10)。
apply 函数的模板为 `[T <: Data]`，这意味着 `RegEnable` 适用于所有 Chisel 硬件类型。

某些操作仅在 `Bits` 的子类型上定义，例如 `+`。
这就是为什么您可以对 `UInt` 或 `SInt` 进行加法运算，但不能对 `Bundle` 或 `Vec` 进行加法运算。

<span style="color:blue">**示例：类型通用移位寄存器**<a name="type-generic-shift-register"></a></span><br>
在 Scala 中，对象和函数并不是我们唯一可以作为参数处理的东西。
我们还可以将类型作为参数处理。

我们通常需要提供类型约束。
在这种情况下，我们希望能够将对象放入 bundle 中，连接它们 (:=)，并使用它们创建寄存器 (RegNext)。
这些操作不能对任意对象执行；例如，wire := 3 是非法的，因为 3 是 Scala Int，而不是 Chisel UInt。
如果我们使用类型约束来说明类型 T 是 Data 的子类，那么我们可以在任何类型为 T 的对象上使用 :=，因为 := 是为所有 Data 定义的。

这是一个将类型作为参数的移位寄存器的实现。
*gen* 是一个类型为 T 的参数，它指示要使用的宽度，例如 new ShiftRegister(UInt(4.W)) 是一个用于 4 位 UInt 的移位寄存器。
*gen* 还允许 Scala 编译器推断类型 T——如果您想更具体，可以编写 new ShiftRegister[UInt](UInt(4.W))，但如果您省略 [UInt]，Scala 编译器也足够聪明来推断它。

In [ ]:
class ShiftRegisterIO[T <: Data](gen: T, n: Int) extends Bundle {
    require (n >= 0, "移位寄存器必须具有非负移位")
    
    val in = Input(gen)
    val out = Output(Vec(n + 1, gen)) // + 1 因为 in 包含在 out 中
    override def cloneType: this.type = (new ShiftRegisterIO(gen, n)).asInstanceOf[this.type]
}

class ShiftRegister[T <: Data](gen: T, n: Int) extends Module {
    val io = IO(new ShiftRegisterIO(gen, n))
    
    io.out.foldLeft(io.in) { case (in, out) =>
        out := in
        RegNext(in)
    }
}

visualize(() => new ShiftRegister(SInt(6.W), 3))
test(new ShiftRegister(SInt(6.W), 3)) { c => 
    println(s"测试类型为 ${c.io.in} 且深度为 ${c.io.out.length} 的 ShiftRegister")
    for (i <- 0 until 10) {
        c.io.in.poke(i.S) // 魔术字面量创建
        println(s"$i: ${c.io.out.indices.map { index => c.io.out(index).peek().litValue} }")
        c.clock.step(1)
    }}

我们通常建议避免将继承与类型泛型一起使用。
正确执行可能非常棘手，并且很快就会令人沮丧。

## 带类型类的类型泛型

上面的示例仅限于可以在任何 `Data` 实例上执行的简单操作，例如 `:=` 或 `RegNext()`。
在生成 DSP 电路时，我们希望执行诸如加法和乘法之类的数学运算。
`dsptools` 库提供了用于编写类型参数化 DSP 生成器的工具。

这是一个编写乘法累加模块的示例。
它可以用于为 `FixedPoint`、`SInt` 甚至 `DspComplex[T]`（`dsptools` 提供的复数类型）生成乘法累加 (MAC)。
类型绑定的语法略有不同，因为 `dsptools` 使用类型类。
它们超出了本笔记本的范围。
有关使用类型类的更多信息，请阅读 `dsptools`自述文件和文档。

`T <: Data : Ring` 表示 `T` 是 `Data` 的子类型，并且也是一个 `Ring`。
`Ring` 在 `dsptools` 中定义为一个具有 `+` 和 `*`（以及其他操作）的数字。

_`Ring` 的替代方案是 `Real`，但这不允许我们为 `DspComplex()` 创建 MAC，因为复数不是 `Real`。_



In [ ]:
import chisel3.experimental._
import dsptools.numbers._

class Mac[T <: Data : Ring](genIn : T, genOut: T) extends Module {
    val io = IO(new Bundle {
        val a = Input(genIn)
        val b = Input(genIn)
        val c = Input(genIn)
        val out = Output(genOut)
    })
    io.out := io.a * io.b + io.c
}

println(getVerilog(new Mac(UInt(4.W), UInt(6.W)) ))
println(getVerilog(new Mac(SInt(4.W), SInt(6.W)) ))
println(getVerilog(new Mac(FixedPoint(4.W, 3.BP), FixedPoint(6.W, 4.BP))))


<span style="color:red">**练习：Mac 作为对象**</span><br>

Mac `Module` 的输入数量很少，只有一个输出。
对于其他 Chisel 生成器来说，编写如下代码可能会很方便：
```scala
val out = Mac(a, b, c)
```

在下面的 `Mac` 伴生对象中实现一个 `apply` 方法，以实现 `Mac` 功能。

In [ ]:
object Mac {
    def apply[T <: Data : Ring](a: T, b: T, c: T): T = {
        ??? // 你的代码
    }
}

class MacTestModule extends Module {
    val io = IO(new Bundle {
        val uin = Input(UInt(4.W))
        val uout = Output(UInt())
        val sin = Input(SInt(4.W))
        val sout = Output(SInt())
        //val fin = Input(FixedPoint(16.W, 12.BP))
        //val fout = Output(FixedPoint())
    })
    // 对于每个 IO 对，执行 out = in * in + in
    io.uout := Mac(io.uin, io.uin, io.uin)
    io.sout := Mac(io.sin, io.sin, io.sin)
    //io.fout := Mac(io.fin, io.fin, io.fin)
}
println(getVerilog(new MacTestModule))

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong>（点击切换显示）</label>
<article>
<pre style="background-color:#f7f7f7">

        a * b + c

</pre></article></div></section></div>

<span style="color:red">**练习：积分器**</span><br>
如下图所示实现一个积分器。$n_1$ 是 `genReg` 的宽度，$n_2$ 是 `genIn` 的宽度。

不要忘记 `Reg`、`RegInit`、`RegNext`、`RegEnable` 等是针对类型 `T <: Data` 进行模板化的。

<img src="images/integrator.svg" alt="积分器" style="width: 250px;"/>

In [ ]:
class Integrator[T <: Data : Ring](genIn: T, genReg: T) extends Module {
    val io = IO(new Bundle {
        val in  = Input(genIn)
        val out = Output(genReg)
    })
    
    ??? // 你的代码
}

test(new Integrator(SInt(4.W), SInt(8.W))) { c =>
    c.io.in.poke(3.S)
    c.io.out.expect(0.S)
    c.clock.step(1)
    c.io.in.poke(-4.S)
    c.io.out.expect(3.S)
    c.clock.step(1)
    c.io.in.poke(6.S)
    c.io.out.expect(-1.S)
    c.clock.step(1)
    c.io.out.expect(5.S)
}

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-2" />
<label for="check-2"><strong>解决方案</strong>（点击切换显示）</label>
<article>
<pre style="background-color:#f7f7f7">

class Integrator\[T <: Data : Ring\](genIn: T, genReg: T) extends Module {
    val io = IO(new Bundle {
        val in  = Input(genIn.cloneType)
        val out = Output(genReg.cloneType)
    })
    
    val reg = RegInit(genReg, Ring[T].zero) // 初始化为零
    reg := reg + io.in
    io.out := reg
}

</pre></article></div></section></div>

---
# 创建自定义类型<a name="creating-a-custom-type"></a>

使 Chisel 强大的原因之一是它的可扩展性。
您可以添加自己的类型，这些类型具有针对您的应用程序定制的自己的操作和表示形式。
本节将介绍创建自定义类型的方法。

<span style="color:blue">**示例：DspComplex**</span><br>
`DspComplex` 是在 **dsptools** 中定义的自定义数据类型 [此处](https://github.com/ucb-bar/dsptools/blob/v1.0.0/src/main/scala/dsptools/numbers/chisel_concrete/DspComplex.scala#L59)。
需要理解的关键行是：
```scala
class DspComplex[T <: Data:Ring](val real: T, val imag: T) extends Bundle { ... }
```

`DspComplex` 是一个类型通用的容器。
这意味着复数的实部和虚部可以是任何类型，只要它们满足由 `T <: Data : Ring` 给出的类型约束即可。

`T <: Data` 表示 `T` 是 `chisel3.Data`（Chisel 对象的基本类型）的子类型。
这意味着 `DspComplex` 仅适用于作为 Chisel 类型而不是任意 Scala 类型的对象。

`T : Ring` 表示存在 `T` 的 Ring 类型类实现。
`Ring` 类型类定义了 `+` 和 `*` 运算符以及加法和乘法单位元（有关环的详细信息，请参阅[此维基百科文章](https://en.wikipedia.org/wiki/Ring_(mathematics))）。
**dsptools** 在[此处](https://github.com/ucb-bar/dsptools/tree/v1.0.0/src/main/scala/dsptools/numbers/chisel_types)为常用的 Chisel 类型定义了类型类。

**dsptools** 还为 `DspComplex` 定义了一个 `Ring` 类型类，因此我们可以将我们的 MAC 生成器与复数一起重用：

In [ ]:
println(getVerilog(new Mac(DspComplex(SInt(4.W), SInt(4.W)), DspComplex(SInt(6.W), SInt(6.W))) ))

<span style="color:red">**练习：符号数值**</span><br>
假设您想使用符号数值表示法，并且希望重用所有 DSP 生成器。
类型类支持这种临时多态性。
以下示例给出了 SignMagnitude 类型实现的开头以及一个 Ring 类型类实现，该实现将允许该类型与 Mac 生成器一起使用。

填写 `+` 和 `*` 的实现。
您应该以 `unary_-()` 的实现为蓝本。
下一个块包含一个测试，用于检查使用 `SignMagnitude` 的 `Mac` 的正确性。

In [ ]:
class SignMagnitude(val magnitudeWidth: Option[Int] = None) extends Bundle {
    val sign = Bool()
    val magnitude = magnitudeWidth match {
        case Some(w) => UInt(w.W)
        case None    => UInt()
    }
    def +(that: SignMagnitude): SignMagnitude = {
        // 实现此函数！
    }
    def -(that: SignMagnitude): SignMagnitude = {
        this.+(-that)
    }
    def unary_-(): SignMagnitude = {
        val result = Wire(new SignMagnitude())
        result.sign := !this.sign
        result.magnitude := this.magnitude
        result
    }
    def *(that: SignMagnitude): SignMagnitude = {
        // 实现此函数！
    }
}
trait SignMagnitudeRing extends Ring[SignMagnitude] {
    def plus(f: SignMagnitude, g: SignMagnitude): SignMagnitude = {
        f + g
    }
    def times(f: SignMagnitude, g: SignMagnitude): SignMagnitude = {
        f * g
    }
    def one: SignMagnitude = {
        val one = Wire(new SignMagnitude(Some(1)))
        one.sign := false.B
        one.magnitude := 1.U
        one
    }
    def zero: SignMagnitude = {
        val zero = Wire(new SignMagnitude(Some(0)))
        zero.sign := false.B
        zero.magnitude := 0.U
        zero
    }
    def negate(f: SignMagnitude): SignMagnitude = {
        -f
    }
    
    // 对于此示例，保持未实现状态
    def minusContext(f: SignMagnitude, g: SignMagnitude): SignMagnitude = ???
    def negateContext(f: SignMagnitude): SignMagnitude = ???
    def plusContext(f: SignMagnitude,g: SignMagnitude): SignMagnitude = ???
    def timesContext(f: SignMagnitude,g: SignMagnitude): SignMagnitude = ???
}
implicit object SignMagnitudeRingImpl extends SignMagnitudeRing

In [ ]:
import chisel3.experimental.BundleLiterals._

test(new Mac(new SignMagnitude(Some(4)), new SignMagnitude(Some(5)))) { c =>
    c.io.a.poke(chiselTypeOf(c.io.a).Lit(_.sign -> false.B, _.magnitude -> 3.U))
    c.io.b.poke(chiselTypeOf(c.io.b).Lit(_.sign -> false.B, _.magnitude -> 3.U))
    c.io.c.poke(chiselTypeOf(c.io.c).Lit(_.sign -> false.B, _.magnitude -> 2.U))
    c.io.out.expect(chiselTypeOf(c.io.out).Lit(_.sign -> false.B, _.magnitude -> 11.U))

    c.io.c.sign.poke(true.B)
    c.io.out.expect(chiselTypeOf(c.io.out).Lit(_.sign -> false.B, _.magnitude -> 7.U))

    c.io.b.sign.poke(true.B)
    c.io.out.expect(chiselTypeOf(c.io.out).Lit(_.sign -> true.B, _.magnitude -> 11.U))
}
println("成功！！") // Scala 代码：如果我们到达这里，我们的测试通过了！

查看 verilog 以确定输出是否合理：

In [ ]:
println(getVerilog(new Mac(new SignMagnitude(Some(4)), new SignMagnitude(Some(5)))))

`SignMagnitude` 甚至可以与 `DspComplex` 一起使用！

In [ ]:
println(getVerilog(new Mac(DspComplex(new SignMagnitude(Some(4)), new SignMagnitude(Some(4))), DspComplex(new SignMagnitude(Some(5)), new SignMagnitude(Some(5))))))

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-3" />
<label for="check-3"><strong>解决方案</strong>（点击切换显示）</label>
<article>
<pre style="background-color:#f7f7f7">
    // SignMagnitude 类的实现

    def +(that: SignMagnitude): SignMagnitude = {
      val result = Wire(new SignMagnitude())
      val signsTheSame = this.sign === that.sign
      when (signsTheSame) {
        result.sign      := this.sign
        result.magnitude := this.magnitude + that.magnitude
      } .otherwise {
        when (this.magnitude > that.magnitude) {
          result.sign      := this.sign
          result.magnitude := this.magnitude - that.magnitude
        } .otherwise {
          result.sign      := that.sign
          result.magnitude := that.magnitude - this.magnitude
        }   
      }   
      result
    }
    def *(that: SignMagnitude): SignMagnitude = {
        val result = Wire(new SignMagnitude())
        result.sign := this.sign ^ that.sign
        result.magnitude := this.magnitude * that.magnitude
        result
    }


</pre></article></div></section></div>